In [ ]:
#| default_exp edit_interactive

## Edit-interactive plan execution

Run a tiny Lisette agent loop against one notebook at a time. The inner agent receives the user plan plus a single compact notebook view, then can mutate only that notebook through scoped tools.

Sometimes a user wants an agent to carry out a bounded notebook edit rather than manually choose each `write_nb` or `update_cell` call. This notebook builds that inner edit loop: one notebook, one plan, a small set of notebook-aware tools, and a final diff.

The edit loop is for bounded delegation, not open-ended repository work. Its job is to give an inner agent a stable notebook view, a small tool belt, and revision-aware feedback so a single notebook can be edited and reviewed without exposing raw notebook JSON.

```python
execute_plan("nbs/02_write.ipynb", "Add an example after the write_nb docs", max_steps=4)
```

### Production contract

The edit-interactive loop is experimental. It stays out of the production core unless it has focused contract tests for bounded scope, stable notebook views, deterministic tool results, final diffs, and clear failure behavior when an inner edit cannot be completed.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import nbskill.edit_interactive as ei
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_tmp_nb, write_nb as _write_tmp_nb
from nbskill.edit_interactive import notebook_view as _example_notebook_view
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
with write_demo_notebook("08_edit_interactive_example.ipynb") as path:
    _write_tmp_nb(new_nb([
        mk_cell("#| default_exp demo"),
        mk_cell("#| export\ndef answer():\n    return 42"),
    ]), path)
    print("\n".join(_example_notebook_view(path).splitlines()[:6]))

In [ ]:
#| export
import ast
import difflib
import json
import os
import re
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import redirect_stderr, redirect_stdout
from dataclasses import dataclass, field
from io import StringIO
from pathlib import Path
from threading import Lock
from fastcore.nbio import mk_cell
from fastcore.nbio import read_nb
from fastcore.nbio import write_nb
from nbdev.doclinks import nbdev_export as _run_nb_export
from nbskill.foundation import (
    cap_text, cell_source, clear_outputs, exported_py_path, parse_one_cell,
    stamp_export_metadata, stamp_notebook_metadata, validate_code_cells,
)
from nbskill.graph import symbol_usage_summary
from nbskill.knowledge import reference_query
from nbskill.parallel import notebook_locks
from nbskill.review import diff_nb

_CAPTURE_LOCK = Lock()

### The inner-agent contract

The system prompt is intentionally narrow. The inner agent edits exactly one notebook, uses only the provided tools, prefers stable cell ids, and stops with a summary when the plan is done.

In [ ]:
#| export
EDIT_INTERACTIVE_SYSTEM = """You are an nbskill notebook-editing subagent.
You edit exactly one notebook in one repository. You receive the project
description, a focused knowledge summary, optional caller/callee impact for the
symbols in scope, and one notebook view. There is no project-scope lookup tool
because that scope is injected into this prompt.

Use only the provided tools: str_replace, edit_cell, add_cell, delete_cell,
execute_cell, and query_knowledge. Keep the work scoped to one small specific
task.

For new behavior, use this loop:
1. Experiment: write the smallest code that explores the idea, execute it, and
   inspect the result before treating it as correct.
2. Function: turn the working experiment into a focused function. Add a Markdown
   cell directly above the exported code explaining why the function is needed
   and why it is useful.
3. Example: add an example cell directly below the function showing how it works.
   If the example is slow or produces artifacts, add `#| eval: false`.
4. Test: add a focused test cell directly below the example so future changes
   keep the same output.

Use execute_cell normally to continue from the last executed cell through the
target cell, like a live notebook kernel. Pass rerun_all=True when earlier cells
changed and the whole notebook state must be rebuilt. Exceptions are returned to
you as tool output; inspect them, edit, and execute again. Stop as soon as the
requested notebook change is complete. Your final message must summarize what
changed, what was executed, and what could not be completed.
"""

In [ ]:
#| export
def capture_call_text(func, **kwargs):
    "Run `func` and return captured stdout/stderr, or the return value."
    out, err = StringIO(), StringIO()
    with _CAPTURE_LOCK, redirect_stdout(out), redirect_stderr(err):
        result = func(**kwargs)
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(str(result))
    return "\n".join(chunk for chunk in chunks if chunk)

### A stable notebook view

The edit loop needs a text representation that is compact enough for a model but precise enough for safe edits. `notebook_view` includes ids, cell types, and full cell source.

In [ ]:
#| export
def notebook_view(path, revision=0):
    "Render one notebook as a compact, stable text view."
    path = Path(path)
    with notebook_locks(path):
        nb = read_nb(path)
        lines = [f"Notebook: {path}", f"Revision: {revision}", ""]
        for idx, cell in enumerate(nb.cells):
            lines.append(f"CELL {idx} id={cell.id} type={cell.cell_type}")
            lines.append("<<<SOURCE")
            lines.append(cell_source(cell).rstrip())
            lines.append("SOURCE")
            lines.append("")
        return "\n".join(lines).rstrip() + "\n"

### Session state

`EditSession` tracks the notebook path, revision, timeout, and operation logs. The revision count makes it clear which tool calls changed the notebook during a plan.

In [ ]:
#| export
@dataclass
class EditSession:
    "Mutable state for one edit-interactive notebook run."
    path: Path
    timeout: int = 30
    revision: int = 0
    agent_id: str | None = None
    log_path: Path | None = None
    log: list[str] = field(default_factory=list)
    tool_log: list[str] = field(default_factory=list)
    history: list[dict] = field(default_factory=list)
    messages: list[dict] = field(default_factory=list)
    live_ns: dict = field(default_factory=lambda: {"__name__": "__main__"})
    executed_until_idx: int = -1
    chat: object | None = None
    notebook_msg_idx: int | None = None
    def __post_init__(self):
        self.path = Path(self.path)
        if self.agent_id is None: self.agent_id = _agent_notebook_id(self.path)
        if self.log_path is None: self.log_path = Path("log") / f"agent-{self.agent_id}.log"
    def reset_live_state(self):
        "Reset the live notebook execution namespace."
        self.live_ns = {"__name__": "__main__"}
        self.executed_until_idx = -1
    def refresh_view(self):
        "Replace the notebook-view message in the Lisette history."
        if self.chat is None or self.notebook_msg_idx is None: return
        msg = self.chat.hist[self.notebook_msg_idx]
        view = notebook_view(self.path, self.revision)
        if isinstance(msg, dict): msg["content"] = view
        else: msg.content = view
    def record_message(self, role, content):
        "Append one agent message to memory and the run log."
        item = {"revision": self.revision, "role": role, "content": str(content)}
        self.messages.append(item)
        _append_agent_log(self, "message", item)
    def record(self, message):
        "Append an operation to the session log."
        self.log.append(f"r{self.revision}: {message}")
        _append_agent_log(self, "operation", {"revision": self.revision, "message": message})
    def record_tool(self, name, detail=""):
        "Append one tool use to the session history."
        suffix = f"({detail})" if detail else "()"
        self.tool_log.append(f"r{self.revision}: {name}{suffix}")
        item = {"revision": self.revision, "tool": name}
        if detail: item["detail"] = detail
        self.history.append(item)
        _append_agent_log(self, "tool", item)

In [ ]:
#| export
def _export_notebook(nb, nb_path):
    py_path = exported_py_path(nb_path, nb)
    if py_path is None: return None
    _run_nb_export(path=str(nb_path))
    if py_path.exists():
        stamp_export_metadata(nb, py_path)
        write_nb(nb, nb_path)
    return py_path


def _save_notebook(nb, path):
    with notebook_locks(path):
        stamp_notebook_metadata(nb)
        write_nb(nb, path)
        _export_notebook(nb, path)

In [ ]:
#| export
def _none_if_blank(value):
    if value is None: return None
    value = str(value)
    return None if value.strip().lower() in {"", "none", "null"} else value

In [ ]:
#| export
def _edit_find_cell_by_id(cells, cell_id):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if getattr(cell, "id", None) == str(cell_id)]
    if len(matches) == 1: return matches[0]
    if not matches: raise ValueError(f"No cell has id {cell_id!r}")
    raise ValueError(f"Multiple cells have id {cell_id!r}")

In [ ]:
#| export
def _source_diff(before, after, label):
    diff = difflib.unified_diff(
        before.splitlines(True), after.splitlines(True),
        fromfile=f"{label}:before", tofile=f"{label}:after",
    )
    text = "".join(diff).strip()
    return text or "No source changes"

In [ ]:
#| export
def _finish_write(session, nb, message, diff):
    _save_notebook(nb, session.path)
    session.revision += 1
    session.record(message)
    session.refresh_view()
    return f"{message}\nrevision={session.revision}\n\n{diff}"

In [ ]:
#| export
def _validate_cell(cell):
    validate_code_cells([cell])
    return cell

In [ ]:
#| export
def _agent_notebook_id(path):
    rel = Path(path).with_suffix("").as_posix()
    rel = re.sub(r"[^A-Za-z0-9_.-]+", "-", rel).strip("-.")
    return rel or "notebook"

In [ ]:
#| export
def _append_agent_log(session, kind, payload):
    path = Path(session.log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    record = {"kind": kind, "agent_id": session.agent_id, "notebook": str(session.path), "payload": payload}
    path.open("a", encoding="utf-8").write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
#| export
def _project_description(path):
    start = Path(path).resolve().parent
    for root in [start, *start.parents]:
        readme = root / "README.md"
        if readme.exists(): return cap_text(readme.read_text(encoding="utf-8"), 2400)
    return "No README.md project description was found."

In [ ]:
#| export
def _knowledge_context(query, top_k=3):
    try: result = reference_query(query, top_k=top_k, current_repo=".")
    except BaseException as exc: return f"Knowledge query unavailable: {type(exc).__name__}: {exc}"
    hits = []
    for hit in result.get("hits", []):
        label = ".".join(item for item in [hit.get("module"), hit.get("symbol")] if item)
        source = cap_text(hit.get("source") or hit.get("docstring") or "", 700)
        hits.append(f"- {label or hit.get('path')}: {source}")
    return "\n".join(hits) if hits else "No relevant knowledge hits."

In [ ]:
#| export
def _symbol_impact_context(symbols):
    names = _split_notebooks(symbols)
    if not names: return "No symbols requested for caller/callee impact."
    try: return symbol_usage_summary(".", names)
    except BaseException as exc: return f"Symbol impact unavailable: {type(exc).__name__}: {exc}"

In [ ]:
#| export
def _subagent_context(path, plan, symbols=None):
    return "\n\n".join([
        "Project description:\n" + _project_description(path),
        "Knowledge summary:\n" + _knowledge_context(plan),
        "Caller/callee impact:\n" + _symbol_impact_context(symbols),
    ])

### The editing tool belt

The inner loop only gets four notebook operations: add, edit, run through a cell, and remove. Each operation validates inputs, refreshes the notebook view, and records a diff-like message for the final report.

In [ ]:
#| export
def make_edit_tools(session):
    "Create notebook-scoped tools for one edit-interactive session."
    def str_replace(
        old_str: str,  # Exact text that must occur once in the notebook
        new_str: str,  # Replacement text
    ) -> str:
        "Replace one exact string occurrence anywhere in the current notebook."
        with notebook_locks(session.path):
            nb = read_nb(session.path)
            matches = []
            for idx, cell in enumerate(nb.cells):
                source = cell_source(cell)
                if old_str in source: matches.append((idx, cell, source))
            session.record_tool("str_replace", f"matches={len(matches)}")
            if len(matches) != 1: raise ValueError(f"old_str matched {len(matches)} cells; expected exactly 1")
            idx, cell, before = matches[0]
            after = before.replace(old_str, new_str, 1)
            if cell.cell_type == "code": _validate_cell(mk_cell(after, cell_type="code"))
            cell.source = after
            clear_outputs(cell)
            msg = f"Replaced text in id={cell.id}"
            return _finish_write(session, nb, msg, _source_diff(before, after, f"cell {cell.id}"))
    def edit_cell(
        id: str,  # Cell id to edit
        old_str: str,  # Exact text that must occur once in this cell
        new_str: str,  # Replacement text
    ) -> str:
        "Replace one exact string occurrence inside one cell."
        with notebook_locks(session.path):
            nb = read_nb(session.path)
            idx, cell = _edit_find_cell_by_id(nb.cells, id)
            before = cell_source(cell)
            count = before.count(old_str)
            session.record_tool("edit_cell", f"id={id!r}, matches={count}")
            if count != 1: raise ValueError(f"old_str matched {count} times in id={id}; expected exactly 1")
            after = before.replace(old_str, new_str, 1)
            if cell.cell_type == "code": _validate_cell(mk_cell(after, cell_type="code"))
            cell.source = after
            clear_outputs(cell)
            msg = f"Edited text id={id}"
            return _finish_write(session, nb, msg, _source_diff(before, after, f"cell {id}"))
    def add_cell(
        after_id: str | None = None,  # Cell id to insert after; blank/None appends
        content: str = "",  # New cell source; may start with %%markdown, %%code, or %%raw
    ) -> str:
        "Add one cell to the current notebook."
        with notebook_locks(session.path):
            nb = read_nb(session.path)
            new_cell = _validate_cell(parse_one_cell(content, "code"))
            anchor = _none_if_blank(after_id)
            target = len(nb.cells)
            if anchor is not None:
                idx, _ = _edit_find_cell_by_id(nb.cells, anchor)
                target = idx + 1
            session.record_tool("add_cell", f"after_id={anchor!r}")
            nb.cells.insert(target, new_cell)
            src = cell_source(new_cell)
            where = f"after id={anchor}" if anchor is not None else "at end"
            msg = f"Added cell id={new_cell.id} {where}"
            return _finish_write(session, nb, msg, _source_diff("", src, f"cell {new_cell.id}"))
    def delete_cell(
        id: str,  # Cell id to delete
    ) -> str:
        "Delete one cell from the current notebook."
        with notebook_locks(session.path):
            nb = read_nb(session.path)
            idx, cell = _edit_find_cell_by_id(nb.cells, id)
            session.record_tool("delete_cell", f"id={id!r}")
            before = cell_source(cell)
            del nb.cells[idx]
            msg = f"Deleted cell id={id}"
            return _finish_write(session, nb, msg, _source_diff(before, "", f"cell {id}"))
    def _exec_python(source, filename):
        tree = ast.parse(source, filename=filename, mode="exec")
        if not tree.body: return None
        if not isinstance(tree.body[-1], ast.Expr):
            exec(compile(tree, filename, "exec"), session.live_ns)
            return None
        prefix = ast.Module(body=tree.body[:-1], type_ignores=[])
        expr = ast.Expression(tree.body[-1].value)
        ast.fix_missing_locations(prefix)
        ast.fix_missing_locations(expr)
        if prefix.body: exec(compile(prefix, filename, "exec"), session.live_ns)
        return eval(compile(expr, filename, "eval"), session.live_ns)
    def _eval_false(source):
        return any(re.match(r"#\|\s*eval:\s*false\b", line.strip(), re.I) for line in source.splitlines()[:5])
    def _execute_one(idx, cell):
        source = cell_source(cell)
        header = f"CELL {idx} id={cell.id}"
        if cell.cell_type != "code": return f"{header} status=skipped reason=non-code"
        if _eval_false(source): return f"{header} status=skipped reason=eval-false"
        out, err = StringIO(), StringIO()
        status, display, tb = "ok", None, ""
        filename = f"{session.path}::{cell.id}"
        with _CAPTURE_LOCK, redirect_stdout(out), redirect_stderr(err):
            try: display = _exec_python(source, filename)
            except BaseException:
                status = "error"
                tb = traceback.format_exc()
        chunks = [f"{header} status={status}"]
        if out.getvalue(): chunks.append("stdout:\n" + out.getvalue().rstrip())
        if err.getvalue(): chunks.append("stderr:\n" + err.getvalue().rstrip())
        if display is not None: chunks.append("display:\n" + repr(display))
        if tb: chunks.append("traceback:\n" + tb.rstrip())
        return "\n".join(chunks)
    def execute_cell(
        id: str,  # Cell id to execute through
        rerun_all: bool = False,  # Reset state and execute from the first cell through id
    ) -> str:
        "Execute through one cell in the live notebook state."
        with notebook_locks(session.path):
            nb = read_nb(session.path)
            target_idx, _ = _edit_find_cell_by_id(nb.cells, id)
            cells = list(nb.cells)
        if rerun_all: session.reset_live_state()
        start = session.executed_until_idx + 1
        if target_idx < start: start = target_idx
        session.record_tool("execute_cell", f"id={id!r}, rerun_all={rerun_all}, start={start}, target={target_idx}")
        reports, status = [], "ok"
        for idx in range(start, target_idx + 1):
            report = _execute_one(idx, cells[idx])
            reports.append(report)
            session.executed_until_idx = idx
            if " status=error" in report:
                status = "error"
                break
        if not reports:
            reports.append(f"CELL {target_idx} id={id} status=skipped reason=already-executed")
        session.record(f"Executed cells {start}:{target_idx} ({status})")
        return "\n\n".join([f"status={status}", *reports])
    def query_knowledge(
        query: str,  # Focused implementation or project-memory query
        top_k: int = 3,  # Number of knowledge hits
    ) -> str:
        "Search the reference knowledge base."
        session.record_tool("query_knowledge", f"top_k={top_k}")
        return _knowledge_context(query, top_k=top_k)
    return [str_replace, edit_cell, add_cell, delete_cell, execute_cell, query_knowledge]

In [ ]:
#| export
def make_chat(model, tools, hist, system_prompt=EDIT_INTERACTIVE_SYSTEM):
    "Create the Lisette chat object for edit-interactive."
    from lisette import Chat
    return Chat(model, sp=system_prompt, tools=tools, hist=hist, stream=str(model).startswith("chatgpt/"))

In [ ]:
#| export
def response_text(response):
    "Extract readable text from a Lisette response or response list."
    if isinstance(response, list) and response: response = response[-1]
    try:
        message = response.choices[0].message
        content = message.content
    except (AttributeError, IndexError, TypeError):
        return "" if response is None else str(response)
    if isinstance(content, list):
        return "".join(str(item.get("text", item)) if isinstance(item, dict) else str(item) for item in content)
    return "" if content is None else str(content)


def plan_result_text(result):
    "Return the human-readable text for an execute_plan result."
    if not isinstance(result, dict): return "" if result is None else str(result)
    if result.get("text"): return str(result["text"])
    sections = [
        "edit-interactive complete",
        "",
        "Final response:",
        str(result.get("summary") or "(no final response)"),
        "",
        "Tools used:",
        "\n".join(item.get("detail", item.get("tool", "")) for item in result.get("history", [])) or "(no tool calls)",
    ]
    return "\n".join(sections).rstrip()

In [ ]:
#| export
def final_diff(path):
    "Return a nbdev code-cell diff, or a clear unavailable message."
    try:
        with notebook_locks(path):
            return capture_call_text(diff_nb, path=str(path))
    except BaseException as exc:
        detail = str(exc)
        if "Could not find notebook" in detail or "No git repository found" in detail:
            return (
                "Code-cell diff against HEAD is unavailable because this notebook has no git baseline. "
                "This is expected for new or untracked notebooks."
            )
        return f"Code-cell diff unavailable: {type(exc).__name__}: {exc}"

### Running a plan

`execute_plan` packages the user's plan, the current notebook view, and the notebook tools into a Lisette chat. The result includes the final answer, tool log, operation log, revision, and code-cell diff.

In [ ]:
#| export
def execute_plan(
    notebook: str,  # Path to the one notebook the inner agent may edit
    plan: str,  # Plan for the inner agent to execute
    model: str | None = None,  # Lisette/LiteLLM model; defaults via NBSKILL_AGENT then a Codex model
    max_steps: int = 8,  # Maximum Lisette tool-loop steps
    timeout: int = 30,  # Reserved for external execution callers
    dry_run: bool = False,  # Return proposal context without invoking the inner agent
    symbols: str | None = None,  # Optional comma-separated symbols for caller/callee impact
    injected_context: str | None = None,  # Prebuilt project context for framework-launched subagents
) -> dict:
    "Execute `plan` against one notebook using a bounded subagent loop."
    path = Path(notebook)
    if not path.exists(): raise ValueError(f"Notebook does not exist: {notebook}")
    model = model or os.environ.get("NBSKILL_AGENT") or "chatgpt/gpt-5.4-mini"
    project_context = injected_context or _subagent_context(path, plan, symbols=symbols)
    initial_view = notebook_view(path, revision=0)
    if dry_run:
        text = "\n".join([
            "notebook subagent dry run",
            "",
            f"Notebook: {path}",
            f"Model: {model}",
            f"Max steps: {max_steps}",
            "Plan:",
            plan,
            "",
            "Injected project context:",
            project_context,
            "",
            "Initial notebook view:",
            initial_view,
        ]).rstrip()
        return {"summary": "Dry run only; no subagent was invoked and no notebook edits were made.", "history": [], "text": text}
    session = EditSession(path=path, timeout=timeout)
    hist = [
        {"role": "user", "content": "Injected project context:\n" + project_context},
        {"role": "user", "content": f"Plan:\n{plan}"},
        {"role": "user", "content": initial_view},
    ]
    for item in hist: session.record_message(item["role"], item["content"])
    tools = make_edit_tools(session)
    chat = make_chat(model, tools=tools, hist=hist)
    session.chat = chat
    session.notebook_msg_idx = len(chat.hist) - 1
    session.refresh_view()
    prompt = (
        "Execute the plan with the experiment, function, example, and test loop from the system prompt. "
        "Use execute_cell to validate the experiment and the final example or test when practical. "
        "Stop when the notebook change is complete."
    )
    session.record_message("user", prompt)
    try:
        result = chat(prompt, max_steps=max_steps, return_all=True)
    except BaseException as exc:
        result = f"notebook subagent failed: {type(exc).__name__}: {exc}"
    if not isinstance(result, (list, str, bytes, dict)) and hasattr(result, "__next__"):
        result = list(result)
    summary = response_text(result).strip() or "(no final response)"
    session.record_message("assistant", summary)
    _save_notebook(read_nb(path), path)
    session.record("Exported notebook after subagent run")
    diff = final_diff(path).strip()
    text = "\n".join([
        "notebook subagent complete",
        "",
        "Final response:",
        summary,
        "",
        "Tools used:",
        "\n".join(session.tool_log) if session.tool_log else "(no tool calls)",
        "",
        "Operation log:",
        "\n".join(session.log) if session.log else "(no notebook operations)",
        "",
        f"Final revision: {session.revision}",
        f"Agent log: {session.log_path}",
        "",
        "Notebook diff:",
        diff,
    ]).rstrip()
    return {
        "summary": summary,
        "history": session.history,
        "messages": session.messages,
        "text": text,
        "model": model,
        "notebook": str(path),
        "revision": session.revision,
        "operations": list(session.log),
        "log_path": str(session.log_path),
        "diff": diff,
    }

In [ ]:
#| export
def _split_notebooks(notebooks):
    if notebooks is None: return []
    if isinstance(notebooks, (list, tuple, set)): return [str(item) for item in notebooks if str(item).strip()]
    return [item.strip() for item in str(notebooks).split(",") if item.strip()]

In [ ]:
#| export
def execute_project_plan(
    plan: str,  # Broad project plan to split into notebook-scoped executions
    notebooks: str | None = None,  # Comma-separated notebooks to target
    model: str | None = None,  # Lisette/LiteLLM model
    max_steps: int = 8,  # Maximum steps per notebook subagent
    timeout: int = 30,  # Reserved for callers that enforce per-step timeouts
    dry_run: bool = True,  # Return per-notebook proposals without mutation by default
    symbols: str | None = None,  # Optional comma-separated symbols for caller/callee impact
) -> str:
    "Launch notebook-scoped subagents for a broad project plan."
    targets = _split_notebooks(notebooks)
    if not targets: raise ValueError("Pass one or more notebooks to execute_project_plan.")
    seen = set()
    repeated = sorted({path for path in targets if path in seen or seen.add(path)})
    if repeated: raise ValueError(f"Duplicate notebook target(s): {', '.join(repeated)}")
    chunks = ["project subagent coordinator", "", f"Dry run: {dry_run}", f"Targets: {len(targets)}"]
    shared_context = _subagent_context(targets[0], plan, symbols=symbols)

    def _run_target(notebook):
        subplan = f"{plan}\n\nScope: edit only {notebook}."
        return execute_plan(
            notebook=notebook, plan=subplan, model=model, max_steps=max_steps,
            timeout=timeout, dry_run=dry_run, symbols=symbols, injected_context=shared_context,
        )

    if len(targets) == 1:
        results = {targets[0]: _run_target(targets[0])}
    else:
        results = {}
        with ThreadPoolExecutor(max_workers=len(targets)) as pool:
            futures = {pool.submit(_run_target, notebook): notebook for notebook in targets}
            for future in as_completed(futures):
                results[futures[future]] = future.result()
    for notebook in targets:
        chunks.extend(["", f"## {notebook}", plan_result_text(results[notebook])])
    return "\n".join(chunks).rstrip()

In [ ]:
with write_demo_notebook("08_edit_view.ipynb") as path:
    _write_tmp_nb(new_nb([
        mk_cell("#| default_exp sample"),
        mk_cell("#| export\ndef public():\n    return 1"),
        mk_cell("assert public() == 1"),
    ]), path)
    view = ei.notebook_view(path, revision=3)
    assert "Revision: 3" in view
    assert "type=code" in view
    assert "type=code" in view
    assert "public()" in view

In [ ]:
with write_demo_notebook("08_edit_tools.ipynb") as path:
    first = mk_cell("#| default_exp sample")
    second = mk_cell("x = 1")
    _write_tmp_nb(new_nb([first, second]), path)
    session = ei.EditSession(path=path)
    class FakeChat:
        def __init__(self): self.hist = [{"role": "user", "content": "plan"}, {"role": "user", "content": ei.notebook_view(path)}]
    session.chat = FakeChat()
    session.notebook_msg_idx = 1
    str_replace, edit_cell, add_cell, delete_cell, execute_cell, query_knowledge = ei.make_edit_tools(session)
    replaced = str_replace("x = 1", "x = 10")
    assert "Replaced text" in replaced
    added = add_cell(second.id, "y = 2")
    nb = _read_tmp_nb(path)
    assert "Added cell" in added
    assert len(nb.cells) == 3
    assert "y = 2" in session.chat.hist[1]["content"]
    new_id = nb.cells[-1].id
    edited = edit_cell(new_id, "2", "3")
    assert "Edited text" in edited
    nb = _read_tmp_nb(path)
    assert nb.cells[-1].source == "y = 3"
    removed = delete_cell(new_id)
    assert "Deleted cell" in removed
    assert len(_read_tmp_nb(path).cells) == 2
    assert session.revision == 4
    assert session.log_path.exists()

In [ ]:
with write_demo_notebook("08_edit_update.ipynb") as path:
    cell = mk_cell("x = 1")
    _write_tmp_nb(new_nb([cell]), path)
    session = ei.EditSession(path=path)
    edit_cell = ei.make_edit_tools(session)[1]
    result = edit_cell(cell.id, "1", "2")
    assert f"Edited text id={cell.id}" in result
    assert read_nb(path).cells[0].source == "x = 2"

In [ ]:
with write_demo_notebook("08_edit_missing.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("x = 1")]), path)
    session = ei.EditSession(path=path)
    delete_cell = ei.make_edit_tools(session)[3]
    try:
        delete_cell("missing")
    except ValueError as exc:
        assert "No cell has id" in str(exc)
    else:
        raise AssertionError("expected missing cell failure")

In [ ]:
with write_demo_notebook("08_edit_duplicate.ipynb") as path:
    _write_tmp_nb(new_nb([mk_cell("x = 1"), mk_cell("x = 1")]), path)
    session = ei.EditSession(path=path)
    str_replace = ei.make_edit_tools(session)[0]
    try:
        str_replace("x = 1", "x = 2")
    except ValueError as exc:
        assert "matched 2 cells" in str(exc)
    else:
        raise AssertionError("expected ambiguous replace failure")

In [ ]:
#| hide
from contextlib import contextmanager


@contextmanager
def _patched_make_chat(fake):
    old_make_chat = ei.make_chat
    ei.make_chat = fake
    try: yield
    finally: ei.make_chat = old_make_chat


class FakeChat:
    last = None
    def __init__(self, model, sp, tools, hist):
        self.model, self.sp, self.tools, self.hist = model, sp, tools, hist
        FakeChat.last = self
    def __call__(self, msg, max_steps=20, return_all=False):
        add_cell = self.tools[2]
        execute_cell = self.tools[4]
        add_cell(None, "answer = 42")
        new_id = read_nb(path).cells[-1].id
        report = execute_cell(new_id)
        assert "status=ok" in report
        return "done; could not run extra checks"


def fake_make_chat(model, tools, hist, system_prompt=ei.EDIT_INTERACTIVE_SYSTEM): return FakeChat(model, system_prompt, tools, hist)


with write_demo_notebook("08_edit_plan.ipynb") as path:
    absolute_path = path.resolve()
    _write_tmp_nb(new_nb([mk_cell("#| default_exp sample")]), absolute_path)
    agent_args = dict(model="fake", max_steps=2, timeout=2, injected_context="test context")
    with _patched_make_chat(fake_make_chat): result = ei.execute_plan(str(absolute_path), "Add an answer cell.", **agent_args)
    text = ei.plan_result_text(result)

assert result["summary"] == "done; could not run extra checks"
assert [item["tool"] for item in result["history"]] == ["add_cell", "execute_cell"]
assert result["revision"] == 1
assert result["log_path"].endswith(".log")
assert "Final response:" in text
assert "Tools used:" in text
assert "Agent log:" in text
assert "add_cell" in text
assert "execute_cell" in text
assert "done" in text
assert FakeChat.last.hist[2]["content"].count("answer = 42") == 1